# Liver Disease Prediction for Beginners

Hi! In this notebook I am trying to predict whether a person has liver disease or not based on simple questions about their lifestyle and health habits.

Since real liver disease data is hard to find with only simple questions (usually they require blood tests), I am generating my own "synthetic" (fake but realistic) data. I will simulate 10,000 patients based on actual medical statistics so that our model can learn from realistic data!

**Goal:** Predict if someone has liver disease based on simple questions like age, alcohol use, and symptoms.

## Importing Libraries

First I will import all the libraries that I need to work with the data and visualize it.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.utils import resample
import pickle
import os

## 1. Creating the Fake Data

Let me create realistic random numbers for 10,000 people. I'll include things like age, BMI, alcohol habits, and whether they have symptoms like jaundice or fatigue.

In [ ]:
N = 10_000  # We will pretend we asked 10,000 people

# ── 1. Basic Information ──────────────────────────────────────────────────
age = np.random.randint(18, 85, N)
sex = np.random.binomial(1, 0.70, N)          # 1 = Male, 0 = Female (70% male)
bmi = np.random.normal(26.5, 5.5, N).clip(15, 55)

# ── 2. Lifestyle Habits ───────────────────────────────────────────────────
# Older males tend to drink more on average. 1 = Yes, 0 = No
alcohol_base = 0.20 + 0.15*sex + 0.003*age
alcohol = np.random.binomial(1, alcohol_base.clip(0.05, 0.75), N)

smoker = np.random.binomial(1, 0.30 + 0.10*sex, N)
diet = np.random.normal(5.5, 2.0, N).clip(0, 10)  # Diet score from 0 to 10
diet = diet - 0.5*alcohol  # People who drink might have a slightly worse diet
phys_active = np.random.binomial(1, 0.45 - 0.05*alcohol + 0.01*(10-age//10).clip(0,5), N)

# ── 3. Symptoms and Existing Conditions ───────────────────────────────────
# People who drink alcohol or are older might have more symptoms
jaundice        = np.random.binomial(1, (0.05 + 0.20*alcohol + 0.01*(age>45)).clip(0,0.60), N)
fatigue         = np.random.binomial(1, (0.15 + 0.15*alcohol + 0.05*(bmi>30)).clip(0,0.70), N)
nausea          = np.random.binomial(1, (0.08 + 0.12*alcohol + 0.03*jaundice).clip(0,0.55), N)
abdominal_pain  = np.random.binomial(1, (0.10 + 0.10*alcohol + 0.08*jaundice).clip(0,0.60), N)
loss_appetite   = np.random.binomial(1, (0.07 + 0.08*alcohol + 0.06*jaundice).clip(0,0.50), N)
dark_urine      = np.random.binomial(1, (0.04 + 0.15*jaundice + 0.05*alcohol).clip(0,0.55), N)

diabetes        = np.random.binomial(1, (0.10 + 0.008*age + 0.03*(bmi>30)).clip(0,0.50), N)
hypertension    = np.random.binomial(1, (0.10 + 0.006*age + 0.04*(bmi>30)).clip(0,0.55), N)
family_history  = np.random.binomial(1, 0.15, N)

print(f"Generated {N} fake patients successfully!")

## 2. Deciding Who Has Liver Disease

I will use some basic math (based on medical research) to figure out who should have liver disease based on the answers they gave above! Then I'll put everything into a nice table (DataFrame).

In [ ]:
# Here is the math that calculate a risk score for liver disease!
log_odds = (
    -2.2                              
    + 1.25  * alcohol                 
    + 0.015 * age                     
    + 0.60  * (bmi > 30).astype(int)  
    + 0.50  * (bmi > 35).astype(int)  
    + 0.80  * sex                     
    + 0.40  * smoker                  
    + 1.80  * jaundice                
    + 0.70  * fatigue                 
    + 0.55  * nausea                  
    + 0.65  * abdominal_pain          
    + 0.60  * loss_appetite           
    + 0.90  * dark_urine              
    + 0.45  * diabetes                
    + 0.35  * hypertension            
    + 0.30  * family_history          
    - 0.03  * diet                    
    - 0.20  * phys_active             
    + np.random.normal(0, 0.4, N)     # Add some randomness
)

# Convert the math score into a percentage (probability)
prob = 1 / (1 + np.exp(-log_odds))

# If the probability is more than 50%, we label them as having Liver Disease (1)
target = (prob > 0.50).astype(int)

# Let's put all the data together into a table!
df = pd.DataFrame({
    'Age'           : age,
    'Sex'           : sex,           # 1=Male, 0=Female
    'BMI'           : bmi.round(1),
    'Alcohol'       : alcohol,       # 1=Yes, 0=No
    'Smoker'        : smoker,
    'PhysActive'    : phys_active,
    'DietScore'     : diet.round(1),
    'Diabetes'      : diabetes,
    'Hypertension'  : hypertension,
    'FamilyHistory' : family_history,
    'Jaundice'      : jaundice,
    'Fatigue'       : fatigue,
    'Nausea'        : nausea,
    'AbdominalPain' : abdominal_pain,
    'LossOfAppetite': loss_appetite,
    'DarkUrine'     : dark_urine,
    'LiverDisease'  : target
})

print('Dataset shape:', df.shape)
df.head(3)

## 3. Exploratory Data Analysis (EDA)

Now I will make some graphs to visually understand the data and see what factors are most common for people with liver disease.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle('Liver Disease - Synthetic Dataset EDA', fontsize=14, fontweight='bold')

# plot 1 - Class balance
df['LiverDisease'].value_counts().plot(kind='bar', ax=axes[0,0],
    color=['mediumseagreen','darkorange'], edgecolor='black', rot=0)
axes[0,0].set_xticklabels(['No Disease','Liver Disease'])
axes[0,0].set_title('Class Distribution')

# plot 2 - BMI
df[df['LiverDisease']==0]['BMI'].hist(bins=30, ax=axes[0,1], alpha=0.7, color='mediumseagreen', label='No')
df[df['LiverDisease']==1]['BMI'].hist(bins=30, ax=axes[0,1], alpha=0.7, color='darkorange', label='Yes')
axes[0,1].set_title('BMI by Outcome')
axes[0,1].legend()

# plot 3 - Age
df[df['LiverDisease']==0]['Age'].hist(bins=20, ax=axes[0,2], alpha=0.7, color='mediumseagreen', label='No')
df[df['LiverDisease']==1]['Age'].hist(bins=20, ax=axes[0,2], alpha=0.7, color='darkorange', label='Yes')
axes[0,2].set_title('Age by Outcome')
axes[0,2].legend()

# plot 4 - Symptom Rates
sym_cols = ['Jaundice','Fatigue','Nausea','AbdominalPain','LossOfAppetite','DarkUrine']
dis_rates = [df[df['LiverDisease']==1][c].mean() for c in sym_cols]
nod_rates = [df[df['LiverDisease']==0][c].mean() for c in sym_cols]
x = np.arange(len(sym_cols))
axes[1,0].bar(x-0.18, nod_rates, 0.35, label='No Disease', color='mediumseagreen', edgecolor='k')
axes[1,0].bar(x+0.18, dis_rates, 0.35, label='Liver Disease', color='darkorange', edgecolor='k')
axes[1,0].set_xticks(x)
axes[1,0].set_xticklabels(sym_cols, rotation=20, ha='right', fontsize=8)
axes[1,0].set_title('Symptom Rates')
axes[1,0].legend()

# plot 5 - Alcohol
alc_rate = df.groupby('Alcohol')['LiverDisease'].mean()
axes[1,1].bar(['No Alcohol','Alcohol'], alc_rate.values, color=['mediumseagreen','darkorange'], edgecolor='k')
axes[1,1].set_title('Liver Disease Rate by Alcohol Use')
axes[1,1].set_ylabel('Disease Rate')

# plot 6 - Correlation
features_list = [c for c in df.columns if c != 'LiverDisease']
corr = df[features_list].corrwith(df['LiverDisease']).abs().sort_values()
corr.plot(kind='barh', ax=axes[1,2], color='darkorange', edgecolor='k')
axes[1,2].set_title('Feature Correlation with Liver Disease')

plt.tight_layout()
plt.savefig('liver_eda.png', bbox_inches='tight')
plt.show()
print("EDA plot saved.")

## 4. Preparing Data for Machine Learning

Before training a model, we need to split our dataset into a 'training set' (to teach the machine) and a 'testing set' (to test if the machine actually learned something).

In [ ]:
features_list = [c for c in df.columns if c != 'LiverDisease']
target_col = 'LiverDisease'

X = df[features_list]
y = df[target_col]

# Split 80% for training and 20% for testing
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y)

# We scale the features so numbers like BMI (20-40) and Age (18-85) don't confuse the model
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print()
print('Training samples (for learning):', X_train.shape[0])
print('Testing samples (for checking):', X_test.shape[0])

## 5. Training the Model

Now I will train a simple Logistic Regression model using the training data so we can save it.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# create and train the model
final_model = LogisticRegression(max_iter=1000)
final_model.fit(X_train_sc, y_train)

# test it
predictions = final_model.predict(X_test_sc)
accuracy = accuracy_score(y_test, predictions)

print(f'Model Accuracy: {accuracy * 100:.2f}%')

## 5. Saving the Model

Finally, I will save the trained model, the scaler, and the list of features so I can use them later in a web application.

In [ ]:
os.makedirs('models', exist_ok=True)

pickle.dump(final_model,  open('models/liver_model.pkl',    'wb'))
pickle.dump(scaler,       open('models/liver_scaler.pkl',   'wb'))
pickle.dump(features_list,open('models/liver_features.pkl', 'wb'))

print('Saved successfully!')
print()
print('Web app inputs for liver disease will need:')
print(' - Age, Sex, BMI')
print(' - Alcohol use (Yes/No)')
print(' - Smoker (Yes/No)')
print(' - Physically active (Yes/No)')
print(' - Diet quality score (0-10)')
print(' - Diabetes, Hypertension, Family history (Yes/No)')
print(' - Symptoms: Jaundice, Fatigue, Nausea, Abdominal Pain, Loss of Appetite, Dark Urine')
print()